# ClubElo workbench (prototype)

**Purpose**: prototype `ClubEloModel`, a standalone match model driven entirely by
external ClubElo ratings rather than this pipeline's own goal/xG history, and compare
it head to head against the shipped `XGRateModel`. See
`notebooks/03-prototypes/03-01-forecast-workbench.ipynb` for the equivalent XGRateModel
workbench -- this notebook mirrors its section structure so the two are directly
comparable.

**This is a standard prototype, not a standing exception** (unlike `03-01`): per
`notebooks/README.md`, it stays, gets folded into `src/plforecast` once proven out, or
gets deleted. Nothing here is wired into `forecasting.py`, `club_aliases.yaml`, or the
model ladder -- that is deliberately a later, separate step.

**A real data-access finding, discovered while scoping this notebook**: the classic
`api.clubelo.com` CSV endpoint the project's design docs assumed (`docs/design.md`,
`docs/data-sources.md`, "a single CSV endpoint") returns `502` and appears
discontinued. `clubelo.com` itself is alive but redesigned: each club's page embeds its
rating history as inline chart data (a Vega-Lite spec) rather than exposing a clean
API, and that embedded history only goes back to about **September 2022** per club --
not the full 2015/16 backfill window every other model in this project is backtested
on. So the RPS comparison below is restricted to the seasons where both models have
data (roughly 2022/23 onward), not the full 11-season window `docs/evaluation.md`
reports for `XGRateModel`. The live forecast comparison in section 3 is unaffected by
this, since it only needs each club's *current* rating.

1. **ClubElo data exploration** -- fetch and cache each club's rating history, verify
   name resolution, sanity-check the core assumption (Elo differential vs. actual goal
   difference) before fitting anything.
2. **Lever workbench** -- fit `ClubEloModel`, run it through the same backtest harness
   as every other model, and compare its score directly against `XGRateModel` on the
   same rows.
3. **Forecast visualization** -- a live `ClubEloModel` forecast, the same final-table
   view `03-01` produces, and a side-by-side comparison against `XGRateModel`'s current
   forecast -- including the promoted-club case ClubElo's Championship coverage is
   expected to help most.


In [ ]:
import json
from pathlib import Path

import httpx
import numpy as np
import plotly.express as px
import polars as pl
from scipy.stats import poisson

from plforecast import viz
from plforecast.artifacts.schema import Provenance
from plforecast.artifacts.writer import (
    build_forecast_document,
    git_provenance,
    now_utc,
    package_versions,
    snapshot_provenance,
)
from plforecast.config import Settings
from plforecast.entities.clubs import load_club_dimension
from plforecast.evaluate.backtest import run_backtest
from plforecast.evaluate.calibration import calibration_by_outcome
from plforecast.evaluate.metrics import outcome_index
from plforecast.evaluate.report import format_table
from plforecast.models.base import (
    ClubId,
    UnknownClubError,
    assert_no_odds_columns,
    drop_odds_columns,
)
from plforecast.models.xg_rates import XGRateModel
from plforecast.simulate.competition import PREMIER_LEAGUE
from plforecast.simulate.engine import simulate_season
from plforecast.simulate.tiebreak import PremierLeagueTiebreaks
from plforecast.storage.db import connect

REPO_ROOT = Path("../..")
NOTEBOOK_CONFIG = Settings(data_dir=REPO_ROOT / "data")
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
EVALUATION_DIR = REPO_ROOT / "docs" / "evaluation"
CLUBELO_CACHE_DIR = REPO_ROOT / "data" / "clubelo-cache"
CLUBELO_CACHE_DIR.mkdir(parents=True, exist_ok=True)

## Section 1: ClubElo data exploration

Fetches and caches each club's rating history directly from `clubelo.com` (prototype
code -- not `ingest/base.py`'s `Source`/snapshot machinery, which is for a real
adapter once this model is proven out). `club_aliases.yaml`'s `clubelo_name` column is
currently `null` for every club (a placeholder); the mapping below is this notebook's
own throwaway guess, found by probing `clubelo.com`'s per-country index pages
(`/ENG`) during scoping. Three clubs could not be resolved this way (Huddersfield,
Stoke, Watford -- all out of the top flight since before ClubElo's available history
starts, so their absence has little effect on the seasons this notebook can actually
compare).

In [ ]:
CLUBELO_SLUGS: dict[str, str | None] = {
    "arsenal": "Arsenal",
    "aston-villa": "AstonVilla",
    "bournemouth": "Bournemouth",
    "brentford": "Brentford",
    "brighton": "Brighton",
    "burnley": "Burnley",
    "cardiff": "Cardiff",
    "chelsea": "Chelsea",
    "coventry": "Coventry",
    "crystal-palace": "CrystalPalace",
    "everton": "Everton",
    "fulham": "Fulham",
    "huddersfield": None,  # unresolved -- see markdown above
    "hull": "Hull",
    "ipswich": "Ipswich",
    "leeds": "Leeds",
    "leicester": "Leicester",
    "liverpool": "Liverpool",
    "luton": "Luton",
    "man-city": "ManCity",
    "man-united": "ManUnited",
    "middlesbrough": "Middlesbrough",
    "newcastle": "Newcastle",
    "norwich": "Norwich",
    "nottm-forest": "Forest",
    "sheffield-united": "SheffieldUnited",
    "southampton": "Southampton",
    "stoke": None,  # unresolved -- see markdown above
    "sunderland": "Sunderland",
    "swansea": "Swansea",
    "tottenham": "Tottenham",
    "watford": None,  # unresolved -- see markdown above
    "west-brom": "WestBrom",
    "west-ham": "WestHam",
    "wolves": "Wolves",
}

In [ ]:
def _extract_vega_json(html: str) -> dict:
    marker = "vegaJson = "
    start = html.index(marker) + len(marker)
    obj, _ = json.JSONDecoder().raw_decode(html, start)
    return obj


def fetch_clubelo_history(club_id: str, slug: str) -> pl.DataFrame:
    """One club's Elo rating history, cached to CLUBELO_CACHE_DIR after the first
    fetch so re-running this cell while tweaking doesn't re-hit the site. Returns an
    empty frame (not an error) for a slug the site doesn't recognise, since a few
    clubs are known to be unresolved (see CLUBELO_SLUGS)."""
    cache_path = CLUBELO_CACHE_DIR / f"{club_id}.json"
    if cache_path.exists():
        rows = json.loads(cache_path.read_text())
    else:
        response = httpx.get(f"https://clubelo.com/{slug}", timeout=15, follow_redirects=True)
        response.raise_for_status()
        spec = _extract_vega_json(response.text)
        rows = []
        for dataset in spec.get("datasets", {}).values():
            if dataset and "Elo" in dataset[0] and "Date" in dataset[0]:
                rows = dataset
                break
        cache_path.write_text(json.dumps(rows))

    if not rows:
        return pl.DataFrame(schema={"date": pl.Date, "elo": pl.Float64, "golo": pl.Float64})
    return (
        pl.DataFrame(rows)
        .select(
            pl.col("Date").str.slice(0, 10).str.to_date().alias("date"),
            pl.col("Elo").alias("elo"),
            pl.col("Golo").alias("golo"),
        )
        .sort("date")
    )


elo_history: dict[str, pl.DataFrame] = {}
for club_id, slug in CLUBELO_SLUGS.items():
    if slug is None:
        continue
    elo_history[club_id] = fetch_clubelo_history(club_id, slug)

coverage = pl.DataFrame(
    [
        {
            "club_id": club_id,
            "rows": frame.height,
            "earliest": frame["date"].min() if frame.height else None,
            "latest": frame["date"].max() if frame.height else None,
        }
        for club_id, frame in elo_history.items()
    ]
).sort("club_id")
coverage

Elo trajectory for a couple of clubs -- a visual sanity check that the fetch and
parse actually worked.

In [ ]:
sample_clubs = [
    c for c in ["arsenal", "man-city", "liverpool"] if elo_history.get(c, pl.DataFrame()).height
]
sample = pl.concat([elo_history[c].with_columns(pl.lit(c).alias("club_id")) for c in sample_clubs])
px.line(sample.to_pandas(), x="date", y="elo", color="club_id", title="ClubElo rating over time")

### Elo differential vs. actual goal difference

The core assumption `ClubEloModel` rests on: does a bigger Elo gap actually predict a
bigger goal difference in this pipeline's own match data? Checked before any fitting
happens.

In [ ]:
conn = connect(NOTEBOOK_CONFIG)
matches = conn.execute(
    "SELECT match_id, season, date, home_club_id, away_club_id, home_goals, away_goals "
    "FROM stg_matches ORDER BY date"
).pl()
conn.close()

elo_long = pl.concat(
    [
        frame.select("date", pl.lit(club_id).alias("club_id"), "elo")
        for club_id, frame in elo_history.items()
        if frame.height
    ]
).sort(["club_id", "date"])

home_elo = (
    matches.select(pl.col("date"), pl.col("home_club_id").alias("club_id"), pl.col("match_id"))
    .sort(["club_id", "date"])
    .join_asof(elo_long, on="date", by="club_id", strategy="backward")
    .rename({"elo": "home_elo"})
    .select("match_id", "home_elo")
)
away_elo = (
    matches.select(pl.col("date"), pl.col("away_club_id").alias("club_id"), pl.col("match_id"))
    .sort(["club_id", "date"])
    .join_asof(elo_long, on="date", by="club_id", strategy="backward")
    .rename({"elo": "away_elo"})
    .select("match_id", "away_elo")
)
elo_matched = (
    matches.join(home_elo, on="match_id")
    .join(away_elo, on="match_id")
    .drop_nulls(["home_elo", "away_elo"])
)
elo_matched = elo_matched.with_columns(
    (pl.col("home_elo") - pl.col("away_elo")).alias("elo_diff"),
    (pl.col("home_goals") - pl.col("away_goals")).alias("goal_diff"),
)
print(f"{elo_matched.height} of {matches.height} matches have both clubs' Elo as of that date")
px.scatter(
    elo_matched.to_pandas(),
    x="elo_diff",
    y="goal_diff",
    opacity=0.3,
    title="Elo differential vs. actual goal difference",
)

### A promoted club across divisions

`ClubElo` is expected to cover a promoted club's Championship history, unlike
Understat. Shown here for whichever of this season's promoted clubs resolved above --
compare against `docs/model-card.md`'s account of the same club's early-season
forecast before the survival-zone prior existed.

In [ ]:
promoted_examples = [
    c
    for c in ["coventry", "hull", "sunderland", "leeds"]
    if elo_history.get(c, pl.DataFrame()).height
]
if promoted_examples:
    club_id = promoted_examples[0]
    px.line(
        elo_history[club_id].to_pandas(),
        x="date",
        y="elo",
        title=f"ClubElo rating over time: {club_id}",
    )
else:
    print("none of the sampled promoted/recently-promoted clubs resolved a ClubElo slug")

## Section 2: Lever workbench

`ClubEloModel`'s attack/defence signal is entirely external (each club's own Elo
rating), so unlike every other rung it has almost nothing to fit *per club* -- only
three global constants (`mu`, home advantage, and the Elo-to-goal-rate slope), by
ordinary least squares on `(elo_diff, log goals)` pairs. This is why the plan's
originally-proposed `HOME_ADVANTAGE_ELO`/`ELO_SCALE` constants aren't levers below:
fitting the home-advantage and Elo-sensitivity terms directly from data, the same way
every other model's parameters are chosen, removes the need to assume them by hand.

In [ ]:
class ClubEloModel:
    """Prototype match model: attack/defence signal comes entirely from external
    ClubElo ratings, not this pipeline's own goal/xG history. `elo_history` maps
    club_id -> a (date, elo) frame sorted by date. `fit()` calibrates mu/home
    advantage/Elo-sensitivity by OLS on (elo_diff, log goals) pairs, using each
    match's own point-in-time Elo differential (asof-joined); `scoreline_matrix()`
    then uses each club's *current* (as-of-fit-date) rating to predict forward --
    the same MatchModel protocol as every other model (models/base.py), so it plugs
    directly into run_backtest/simulate_season/etc. with no new evaluation code."""

    def __init__(
        self, elo_history: dict[str, pl.DataFrame], *, rho: float = 0.0, floor: float = 0.05
    ):
        self.elo_history = {
            club_id: frame for club_id, frame in elo_history.items() if frame.height
        }
        self.rho = rho
        self.floor = floor
        self._mu = 0.0
        self._home_advantage = 0.0
        self._k = 0.0
        self._current_elo: dict[str, float] = {}

    def _elo_long(self) -> pl.DataFrame:
        return pl.concat(
            [
                frame.select("date", pl.lit(club_id).alias("club_id"), "elo")
                for club_id, frame in self.elo_history.items()
            ]
        ).sort(["club_id", "date"])

    def fit(self, matches: pl.DataFrame) -> "ClubEloModel":
        assert_no_odds_columns(matches)
        if not self.elo_history:
            raise ValueError("no ClubElo history loaded -- run section 1 first")
        as_of = matches["date"].max()
        elo_long = self._elo_long()

        home_elo = (
            matches.select("match_id", "date", pl.col("home_club_id").alias("club_id"))
            .sort(["club_id", "date"])
            .join_asof(elo_long, on="date", by="club_id", strategy="backward")
            .rename({"elo": "home_elo"})
            .select("match_id", "home_elo")
        )
        away_elo = (
            matches.select("match_id", "date", pl.col("away_club_id").alias("club_id"))
            .sort(["club_id", "date"])
            .join_asof(elo_long, on="date", by="club_id", strategy="backward")
            .rename({"elo": "away_elo"})
            .select("match_id", "away_elo")
        )
        frame = (
            matches.join(home_elo, on="match_id")
            .join(away_elo, on="match_id")
            .drop_nulls(["home_elo", "away_elo"])
        )
        if frame.height == 0:
            raise ValueError("no matches with resolvable Elo on both sides to fit on")

        elo_diff = (frame["home_elo"] - frame["away_elo"]).to_numpy()
        home_target = np.maximum(frame["home_goals"].cast(pl.Float64).to_numpy(), self.floor)
        away_target = np.maximum(frame["away_goals"].cast(pl.Float64).to_numpy(), self.floor)

        n = frame.height
        design = np.zeros((2 * n, 3))
        target = np.empty(2 * n)
        rows = np.arange(n)
        design[2 * rows, 0] = 1.0
        design[2 * rows, 1] = 1.0
        design[2 * rows, 2] = elo_diff
        target[2 * rows] = np.log(home_target)
        design[2 * rows + 1, 0] = 1.0
        design[2 * rows + 1, 2] = -elo_diff
        target[2 * rows + 1] = np.log(away_target)

        solution, *_ = np.linalg.lstsq(design, target, rcond=None)
        self._mu, self._home_advantage, self._k = (float(v) for v in solution)

        self._current_elo = {}
        for club_id in set(matches["home_club_id"]) | set(matches["away_club_id"]):
            history = self.elo_history.get(club_id)
            if history is None:
                continue
            as_of_rating = history.filter(pl.col("date") <= as_of)
            if as_of_rating.height:
                self._current_elo[club_id] = float(as_of_rating[-1, "elo"])
        return self

    def rates(self, home: ClubId, away: ClubId) -> tuple[float, float]:
        if home not in self._current_elo or away not in self._current_elo:
            unknown = [c for c in (home, away) if c not in self._current_elo]
            raise UnknownClubError(f"club(s) with no resolvable ClubElo rating: {unknown!r}")
        elo_diff = self._current_elo[home] - self._current_elo[away]
        lam_home = np.exp(self._mu + self._home_advantage + self._k * elo_diff)
        lam_away = np.exp(self._mu - self._k * elo_diff)
        return float(lam_home), float(lam_away)

    def scoreline_matrix(self, home: ClubId, away: ClubId, max_goals: int = 10) -> np.ndarray:
        lam_home, lam_away = self.rates(home, away)
        goals = np.arange(max_goals + 1)
        matrix = np.outer(poisson.pmf(goals, lam_home), poisson.pmf(goals, lam_away))
        if self.rho:
            matrix[0, 0] *= 1 - lam_home * lam_away * self.rho
            matrix[0, 1] *= 1 + lam_home * self.rho
            matrix[1, 0] *= 1 + lam_away * self.rho
            matrix[1, 1] *= 1 - self.rho
            matrix = np.clip(matrix, 0.0, None)
        return np.asarray(matrix / matrix.sum())

    @property
    def config_hash(self) -> str:
        import hashlib

        config = {"model": "clubelo-prototype", "rho": self.rho, "floor": self.floor}
        return hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()

In [ ]:
# ---- parameters: edit and re-run this cell and everything below it ----
RHO = 0.0
MIN_TRAIN_MATCHES = 100
CADENCE = "date"  # "date" | "gameweek"
COMPARISON_SEASON_FLOOR = "2022/23"  # ClubElo's available history starts ~Sept 2022

In [ ]:
conn = connect(NOTEBOOK_CONFIG)
evaluation_matches = conn.execute(
    "SELECT match_id, season, date, home_club_id, away_club_id, home_goals, away_goals, "
    "home_xg, away_xg, result, benchmark_home_odds, benchmark_draw_odds, "
    "benchmark_away_odds, benchmark_source "
    "FROM stg_matches WHERE season != (SELECT max(season) FROM stg_matches) ORDER BY date"
).pl()
conn.close()

comparison_matches = evaluation_matches.filter(pl.col("season") >= COMPARISON_SEASON_FLOOR)
print(
    f"comparison window: {comparison_matches['season'].n_unique()} seasons, "
    f"{comparison_matches.height} matches (full evaluation window: "
    f"{evaluation_matches['season'].n_unique()} seasons, {evaluation_matches.height} matches)"
)

clubelo_backtest = run_backtest(
    comparison_matches,
    lambda: ClubEloModel(elo_history, rho=RHO),
    min_train_matches=MIN_TRAIN_MATCHES,
    cadence=CADENCE,
)
xgrate_backtest = run_backtest(
    comparison_matches,
    lambda: XGRateModel(xi=0.005, blend=0.75, rho=0.0),
    min_train_matches=MIN_TRAIN_MATCHES,
    cadence=CADENCE,
)

print(
    f"clubelo scored: {clubelo_backtest.n_scored}, unrateable: {clubelo_backtest.unrateable.height}"
)
print(
    f"xg-rates scored: {xgrate_backtest.n_scored}, unrateable: {xgrate_backtest.unrateable.height}"
)

Direct comparison, same rows both models could price:

In [ ]:
shared_keys = clubelo_backtest.scores.select("season", "date", "home_club_id", "away_club_id").join(
    xgrate_backtest.scores.select("season", "date", "home_club_id", "away_club_id"),
    on=["season", "date", "home_club_id", "away_club_id"],
    how="inner",
)


def _on_shared_rows(scores: pl.DataFrame) -> pl.DataFrame:
    return scores.join(
        shared_keys, on=["season", "date", "home_club_id", "away_club_id"], how="semi"
    )


def _row(name: str, scores: pl.DataFrame) -> dict:
    restricted = _on_shared_rows(scores)
    return {
        "model": name,
        "n": restricted.height,
        "rps": restricted["rps"].mean(),
        "log_loss": restricted["log_loss"].mean(),
        "brier": restricted["brier"].mean(),
    }


print(f"identical rows: {shared_keys.height}")
print(
    format_table(
        [
            _row("clubelo-prototype", clubelo_backtest.scores),
            _row("xg-rates", xgrate_backtest.scores),
        ]
    )
)

Calibration for `ClubEloModel` on the comparison window:

In [ ]:
clubelo_shared = _on_shared_rows(clubelo_backtest.scores)
probs = clubelo_shared.select("p_home", "p_draw", "p_away").to_numpy()
outcomes = outcome_index(clubelo_shared["result"].to_list())
calibration_rows = calibration_by_outcome(probs, outcomes).to_dicts()
viz.calibration_chart(calibration_rows)

Scoreline matrices for a couple of illustrative fixtures, `ClubEloModel` fit on the
full comparison window -- the same "change a lever, see the model change" visual as
`03-01`.

In [ ]:
illustrative_fitted = ClubEloModel(elo_history, rho=RHO).fit(drop_odds_columns(comparison_matches))
illustrative_fixtures = (
    comparison_matches.filter(pl.col("season") == comparison_matches["season"].max())
    .select("home_club_id", "away_club_id")
    .unique()
    .head(2)
)
for home_id, away_id in illustrative_fixtures.iter_rows():
    heatmap = viz.scoreline_heatmap(
        illustrative_fitted.scoreline_matrix(home_id, away_id, max_goals=6),
        home_id,
        away_id,
    )
    heatmap.show()

## Section 3: Forecast visualization

`ClubEloModel` isn't wired into `forecasting.fit_and_simulate` (that's `src/`
production code -- out of scope for this prototype), so this section fits it directly
on every available match and calls `simulate.engine.simulate_season` the same way
`fit_and_simulate` does internally, using the same `stg_fixtures` query `03-01` uses.


In [ ]:
conn = connect(NOTEBOOK_CONFIG)
fixtures = conn.execute(
    "SELECT fixture_id, season, gameweek, kickoff_time, home_club_id, away_club_id, "
    "home_goals, away_goals, finished FROM stg_fixtures ORDER BY kickoff_time, fixture_id"
).pl()
history = conn.execute(
    "SELECT match_id, season, date, home_club_id, away_club_id, home_goals, away_goals, "
    "home_xg, away_xg, result FROM stg_matches ORDER BY date"
).pl()
conn.close()

season_label = str(fixtures["season"][0])
played = fixtures.filter(pl.col("finished"))
remaining = fixtures.filter(~pl.col("finished"))

clubelo_fitted = ClubEloModel(elo_history, rho=RHO).fit(history)
unrateable = [
    (h, a)
    for h, a in remaining.select("home_club_id", "away_club_id").unique().iter_rows()
    if h not in clubelo_fitted._current_elo or a not in clubelo_fitted._current_elo
]
print(f"fixtures with an unrateable club under ClubEloModel: {len(unrateable)}")
if unrateable:
    print(unrateable)

In [ ]:
SIMULATIONS = 5_000
SEED = 42

ratable_remaining = remaining.filter(
    pl.col("home_club_id").is_in(list(clubelo_fitted._current_elo))
    & pl.col("away_club_id").is_in(list(clubelo_fitted._current_elo))
)
clubelo_result = simulate_season(
    played.select("home_club_id", "away_club_id", "home_goals", "away_goals"),
    ratable_remaining.select("home_club_id", "away_club_id"),
    clubelo_fitted,
    PremierLeagueTiebreaks(PREMIER_LEAGUE),
    n_simulations=SIMULATIONS,
    max_goals=10,
    seed=SEED,
)
# doubly-stochastic invariant (design.md 7.3), spot-checked here as in the src test suite
assert (clubelo_result.position_counts.sum(axis=1) == SIMULATIONS).all()
assert (clubelo_result.position_counts.sum(axis=0) == SIMULATIONS).all()
print(f"simulated {len(clubelo_result.club_ids)} clubs, {SIMULATIONS:,} simulations")

### Final projected table (ClubElo)

In [ ]:
dimension = load_club_dimension()
display_names = dict(
    zip(
        dimension.frame["club_id"].to_list(), dimension.frame["display_name"].to_list(), strict=True
    )
)

generated_at = now_utc()
sha, dirty = git_provenance(REPO_ROOT)
provenance = Provenance(
    git_sha=sha,
    git_dirty=dirty,
    model="clubelo-prototype",
    model_config_hash=clubelo_fitted.config_hash,
    simulations=SIMULATIONS,
    random_seed=SEED,
    sources=snapshot_provenance(NOTEBOOK_CONFIG.raw_dir),
    package_versions=package_versions(),
)
clubelo_forecast_doc = build_forecast_document(
    clubelo_result,
    played=played,
    display_names=display_names,
    season=season_label.replace("/", "-"),
    as_of_gameweek=int(played["gameweek"].max() or 0),
    generated_at=generated_at,
    provenance=provenance,
)

clubelo_final_table = pl.DataFrame(
    [
        {
            "pos": i + 1,
            "club": club.display_name,
            "pld": club.current.played,
            "pts": club.current.points,
            "E[pts]": round(club.projected.points.mean, 1),
            "title%": round(club.projected.title * 100, 1),
            "top4%": round(club.projected.top_four * 100, 1),
            "rel%": round(club.projected.relegation * 100, 1),
        }
        for i, club in enumerate(clubelo_forecast_doc.clubs)
    ]
)
clubelo_final_table

In [ ]:
viz.position_matrix(clubelo_result.club_ids, clubelo_result.position_pmf, display_names)

In [ ]:
viz.points_ridgeline(clubelo_result.club_ids, clubelo_result.points, display_names)

### Compare against `XGRateModel`'s current forecast

Runs `forecasting.fit_and_simulate` (the same function `03-01` and `plforecast
forecast` use) for `xg-rates` and joins the two final tables by club.

In [ ]:
from plforecast import forecasting

xgrate_run = forecasting.fit_and_simulate(
    "xg-rates", simulations=SIMULATIONS, seed=SEED, config=NOTEBOOK_CONFIG
)
xgrate_forecast_doc = build_forecast_document(
    xgrate_run.result,
    played=xgrate_run.played,
    display_names=xgrate_run.display_names,
    season=xgrate_run.season_label.replace("/", "-"),
    as_of_gameweek=xgrate_run.as_of_gameweek,
    generated_at=now_utc(),
    provenance=provenance,
)

comparison = (
    clubelo_final_table.select(
        "club", pl.col("pos").alias("clubelo_pos"), pl.col("E[pts]").alias("clubelo_e_pts")
    )
    .join(
        pl.DataFrame(
            [
                {
                    "club": c.display_name,
                    "xgrate_pos": i + 1,
                    "xgrate_e_pts": round(c.projected.points.mean, 1),
                }
                for i, c in enumerate(xgrate_forecast_doc.clubs)
            ]
        ),
        on="club",
        how="full",
        coalesce=True,
    )
    .sort("xgrate_pos")
)
comparison

### Promoted club: ClubElo vs. the survival-zone prior

`XGRateModel`'s forecast for a promoted club leans on `features/priors.py`'s
survival-zone prior (ADR 0006); `ClubEloModel` instead uses that club's real
Championship-era rating, if its ClubElo slug resolved and it fit into the comparison
window's evidence. Compares the two directly for whichever promoted club is available.

In [ ]:
promoted_in_forecast = [c for c in xgrate_run.promoted_clubs if c in clubelo_fitted._current_elo]
if promoted_in_forecast:
    club_id = promoted_in_forecast[0]
    print(f"{club_id}: ClubElo prior-free rating vs. XGRateModel's survival-zone prior")
    promoted_row = comparison.filter(pl.col("club") == display_names.get(club_id, club_id))
else:
    promoted_row = None
    print(
        f"no overlap between this season's promoted clubs {xgrate_run.promoted_clubs} "
        "and resolved ClubElo ratings"
    )
promoted_row